<a href="https://colab.research.google.com/github/bhurlasravan-creator/capstion-project/blob/main/Module_3_%E2%80%94_Support_Assistant_(_support_assistant).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import os
import glob
import json
import subprocess

# Auto-install missing packages for Jupyter / Colab / Standalone environments
REQUIRED_PACKAGES = {
    "chromadb": "chromadb",
    "sentence_transformers": "sentence-transformers",
    "langgraph": "langgraph",
    "pydantic": "pydantic",
    "fastapi": "fastapi",
    "uvicorn": "uvicorn"
}

for module_name, pip_name in REQUIRED_PACKAGES.items():
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing missing dependency: {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

import chromadb
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel, Field
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, END
from fastapi import FastAPI, HTTPException
import uvicorn

CHROMA_PATH = os.environ.get("CHROMA_PATH", "./chroma_db")
COLLECTION_NAME = "zepto_policies"
DOCS_DIR = os.environ.get("DOCS_DIR", "./docs")
MOCK_LLM = os.environ.get("MOCK_LLM", "1") == "1"

# Global model cache to prevent re-downloading/re-loading SentenceTransformer
_MODEL_CACHE = None

def get_embedding_model():
    """Lazy loader for SentenceTransformer model with global memory caching."""
    global _MODEL_CACHE
    if _MODEL_CACHE is None:
        print("⏳ Loading embedding model ('all-MiniLM-L6-v2')... Please wait a few seconds.")
        _MODEL_CACHE = SentenceTransformer("all-MiniLM-L6-v2")
        print("✅ Embedding model loaded into memory!")
    return _MODEL_CACHE

POLICY_DOCS = {
    "doc_01.txt": "Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.",
    "doc_02.txt": "Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto.",
    "doc_03.txt": "Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period.",
    "doc_04.txt": "Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue.",
    "doc_05.txt": "Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee.",
    "doc_06.txt": "If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items without requiring the customer to return the original item, unless the order value exceeds INR 1000, in which case a photo of the issue must be submitted through the report form before a replacement or refund is processed.",
    "doc_07.txt": "Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law.",
    "doc_08.txt": "Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered."
}

def ensure_docs_exist():
    """Auto-generates document directory and writes files if missing."""
    os.makedirs(DOCS_DIR, exist_ok=True)
    for filename, content in POLICY_DOCS.items():
        file_path = os.path.join(DOCS_DIR, filename)
        if not os.path.exists(file_path):
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(content)

def init_vector_store(force_reindex: bool = False):
    """Initializes local SentenceTransformer and populates ChromaDB with smart caching."""
    ensure_docs_exist()
    client = chromadb.PersistentClient(path=CHROMA_PATH)

    # Check if collection already exists and has documents to avoid unnecessary work
    if not force_reindex:
        try:
            col = client.get_collection(COLLECTION_NAME)
            if col.count() >= 8:
                print(f"⚡ Found existing collection '{COLLECTION_NAME}' with {col.count()} documents. Ready!")
                return col
        except Exception:
            pass

    print("🔄 Populating ChromaDB vector store...")
    embedding_model = get_embedding_model()

    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )

    doc_files = sorted(glob.glob(os.path.join(DOCS_DIR, "doc_*.txt")))
    documents, ids, metadatas = [], [], []

    for file_path in doc_files:
        filename = os.path.basename(file_path)
        doc_id = os.path.splitext(filename)[0]

        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read().strip()

        documents.append(content)
        ids.append(doc_id)
        metadatas.append({"source_file": filename, "doc_id": doc_id})

    embeddings = embedding_model.encode(documents).tolist()
    collection.add(documents=documents, embeddings=embeddings, ids=ids, metadatas=metadatas)
    print("✅ Ingestion & Vector indexing complete!")
    return collection

class QueryRequest(BaseModel):
    query: str = Field(..., example="What is the delivery fee for orders under 149?")

class QueryResponse(BaseModel):
    answer: str = Field(..., description="The generated or retrieved response answer")
    sources: List[str] = Field(default_factory=list, description="Document IDs used for answering")
    confidence: float = Field(..., ge=0.0, le=1.0, description="Confidence score between 0.0 and 1.0")

class SupportState(TypedDict):
    query: str
    intent: str
    retrieved_chunks: List[Dict[str, Any]]
    final_response: QueryResponse
    retry_count: int

POLICY_KEYWORDS = [
    "delivery", "return", "refund", "membership",
    "tracking", "cancel", "gift card", "support hours"
]

def classify_intent_node(state: SupportState) -> SupportState:
    query = state["query"].lower()
    is_policy = any(kw in query for kw in POLICY_KEYWORDS)
    state["intent"] = "policy_question" if is_policy else "general_question"
    return state

def route_intent(state: SupportState) -> str:
    return state["intent"]

def retrieve_and_answer_node(state: SupportState) -> SupportState:
    query = state["query"]
    client = chromadb.PersistentClient(path=CHROMA_PATH)

    try:
        collection = client.get_collection(COLLECTION_NAME)
    except Exception:
        collection = init_vector_store()

    embedding_model = get_embedding_model()
    query_vector = embedding_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_vector, n_results=3)

    retrieved_chunks = []
    if results and results["documents"] and len(results["documents"][0]) > 0:
        for doc, doc_id, meta in zip(results["documents"][0], results["ids"][0], results["metadatas"][0]):
            retrieved_chunks.append({"id": doc_id, "content": doc, "metadata": meta})

    state["retrieved_chunks"] = retrieved_chunks
    top_snippet = retrieved_chunks[0]["content"][:200] if retrieved_chunks else "No relevant context found."
    top_id = retrieved_chunks[0]["id"] if retrieved_chunks else "doc_unknown"

    state["final_response"] = QueryResponse(
        answer=f"Based on the retrieved context: {top_snippet}",
        sources=[top_id],
        confidence=1.0
    )
    return state

def direct_answer_node(state: SupportState) -> SupportState:
    state["final_response"] = QueryResponse(
        answer="I can only answer questions about Zepto policies right now.",
        sources=[],
        confidence=1.0
    )
    return state

def build_graph():
    builder = StateGraph(SupportState)
    builder.add_node("classify_intent", classify_intent_node)
    builder.add_node("retrieve_and_answer", retrieve_and_answer_node)
    builder.add_node("direct_answer", direct_answer_node)

    builder.set_entry_point("classify_intent")
    builder.add_conditional_edges(
        "classify_intent",
        route_intent,
        {
            "policy_question": "retrieve_and_answer",
            "general_question": "direct_answer"
        }
    )
    builder.add_edge("retrieve_and_answer", END)
    builder.add_edge("direct_answer", END)
    return builder.compile()

app_graph = build_graph()

app = FastAPI(title="Zepto Support Assistant API", version="1.0.0")

@app.post("/ask", response_model=QueryResponse)
async def ask_question(request: QueryRequest):
    if not request.query or not request.query.strip():
        raise HTTPException(status_code=400, detail="Query string cannot be empty.")

    initial_state = {
        "query": request.query,
        "intent": "",
        "retrieved_chunks": [],
        "final_response": None,
        "retry_count": 0
    }

    final_state = app_graph.invoke(initial_state)
    return final_state["final_response"]

if __name__ == "__main__":
    print("🚀 Initializing Vector Store and Ingesting Documents...")
    init_vector_store()
    print("✅ Ingestion Complete!")

    # Test Queries (runs directly in Jupyter / Notebook / Terminal)
    print("\n--- Test Query 1: Policy Question ---")
    res1 = app_graph.invoke({
        "query": "What is the delivery fee for orders under 149?",
        "intent": "", "retrieved_chunks": [], "final_response": None, "retry_count": 0
    })
    print(json.dumps(res1["final_response"].model_dump(), indent=2))

    print("\n--- Test Query 2: General Question ---")
    res2 = app_graph.invoke({
        "query": "Tell me a software engineering joke.",
        "intent": "", "retrieved_chunks": [], "final_response": None, "retry_count": 0
    })
    print(json.dumps(res2["final_response"].model_dump(), indent=2))

Installing missing dependency: chromadb...


/tmp/ipykernel_480/1015726330.py:117: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  query: str = Field(..., example="What is the delivery fee for orders under 149?")


🚀 Initializing Vector Store and Ingesting Documents...
🔄 Populating ChromaDB vector store...
⏳ Loading embedding model ('all-MiniLM-L6-v2')... Please wait a few seconds.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded into memory!
✅ Ingestion & Vector indexing complete!
✅ Ingestion Complete!

--- Test Query 1: Policy Question ---
{
  "answer": "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del",
  "sources": [
    "doc_01"
  ],
  "confidence": 1.0
}

--- Test Query 2: General Question ---
{
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}
